In [2]:
# ============================================================
# CELL 1 — Imports & Setup
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import spacy
import nltk
from nltk.tokenize import word_tokenize

# Sklearn (for later clustering)
from sklearn.preprocessing import LabelEncoder

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Download NLTK data (only runs once)
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

print("All imports loaded successfully")

✅ All imports loaded successfully


In [11]:
# ============================================================
# CELL 2 — Load Core Datasets (Reddit — 70%)
# ============================================================

import os

DATA_PATH = "../data/processed/"

# --- Load Reddit datasets ---
datasets = {
    "anxiety"      : "reddit_anxiety_cleaned.csv",
    "depression"   : "reddit_depression_cleaned.csv",
    "loneliness"   : "reddit_loneliness_cleaned.csv",
    "mentalhealth" : "reddit_mentalhealth_cleaned.csv",
    "therapy"      : "reddit_therapy_cleaned.csv",
}

dfs = []

for source, filename in datasets.items():
    path = os.path.join(DATA_PATH, filename)
    df = pd.read_csv(path)
    df["source"] = source          # tag where each row came from
    dfs.append(df)
    print(f"✅ {source:15s} → {len(df):>6,} rows | columns: {list(df.columns)}")

# --- Combine into one dataframe ---
reddit_df = pd.concat(dfs, ignore_index=True)

print(f"\n Combined Reddit dataset: {len(reddit_df):,} rows × {reddit_df.shape[1]} columns")
print(f"   Sources: {reddit_df['source'].value_counts().to_dict()}")


✅ anxiety         →    298 rows | columns: ['id', 'source', 'dataset', 'collected_at', 'language', 'cleaned_text', 'rating', 'label', 'emotion']
✅ depression      →    384 rows | columns: ['id', 'source', 'dataset', 'collected_at', 'language', 'cleaned_text', 'rating', 'label', 'emotion']
✅ loneliness      →    139 rows | columns: ['id', 'source', 'dataset', 'collected_at', 'language', 'cleaned_text', 'rating', 'label', 'emotion']
✅ mentalhealth    →    362 rows | columns: ['id', 'source', 'dataset', 'collected_at', 'language', 'cleaned_text', 'rating', 'label', 'emotion']
✅ therapy         →    170 rows | columns: ['id', 'source', 'dataset', 'collected_at', 'language', 'cleaned_text', 'rating', 'label', 'emotion']

 Combined Reddit dataset: 1,353 rows × 9 columns
   Sources: {'depression': 384, 'mentalhealth': 362, 'anxiety': 298, 'therapy': 170, 'loneliness': 139}


In [13]:
df.head()

,id,source,dataset,collected_at,language,cleaned_text,rating,label,emotion
0,1f788068-ff22-4584-99ee-29c72e9e483b,therapy,reddit_therapy,2026-05-02T23:23:51.791529+00:00,en,"Announcement - Important Update on Bot Advertisements Hello, all! We hope you're having all a lo...",NaN,NaN,NaN
1,8e8f54b5-fefc-4c84-99d3-c5777d0138fd,therapy,reddit_therapy,2026-05-02T23:23:51.793528+00:00,en,"Updated Rules Hello, r/therapy! We hope this post finds you well. We are writing to announce a m...",NaN,NaN,NaN
2,f89c07dc-1f71-4e6b-bb61-1d608edabd29,therapy,reddit_therapy,2026-05-02T23:23:51.793528+00:00,en,"""The ax forgets, but the forest remembers"" Recently I re-read the sentiment, ""The axe forgets bu...",NaN,NaN,NaN
3,822a6d67-2ed7-43ee-9cd6-6b81ebe09cf7,therapy,reddit_therapy,2026-05-02T23:23:51.793528+00:00,en,Not sure what type of therapist I'm looking for I've finally decided that I need to go through t...,NaN,NaN,NaN
4,64080621-f500-4f30-9909-d7cef2a8f4bf,therapy,reddit_therapy,2026-05-02T23:23:51.793528+00:00,en,"What should I do? I have a brother who is deaf-mute since birth, he is 26 years old now.. My mom...",NaN,NaN,NaN


In [15]:
# ============================================================
# CELL 3 — Load Support Dataset (Counseling — 20%)
# ============================================================

counseling_df = pd.read_csv(os.path.join(DATA_PATH, "hf_mh_counseling_amod_cleaned.csv"))
counseling_df["source"] = "counseling"

print(f"✅ counseling → {len(counseling_df):,} rows | columns: {list(counseling_df.columns)}")
counseling_df.head(2)

✅ counseling → 830 rows | columns: ['id', 'source', 'dataset', 'collected_at', 'language', 'cleaned_text', 'rating', 'label', 'emotion']


,id,source,dataset,collected_at,language,cleaned_text,rating,label,emotion
0,6aa92e1d-7ccc-4afa-bf2c-0e176b16f712,counseling,mh_counseling_amod,2026-05-03T10:40:00.425566+00:00,en,I'm going through some things with my feelings and myself. I barely sleep and I do nothing but t...,NaN,NaN,NaN
1,d0387c40-1d0d-4aa6-87c7-fb768b238e3b,counseling,mh_counseling_amod,2026-05-03T10:40:00.427774+00:00,en,"I have so many issues to address. I have a history of sexual abuse, I'm a breast cancer survivor...",NaN,NaN,NaN


In [16]:
# ============================================================
# CELL 4 — Explore & Preview
# ============================================================

# --- Reddit overview ---
print("=" * 55)
print("📊 REDDIT DATASETS")
print("=" * 55)
print(reddit_df["source"].value_counts())
print(f"\nTotal rows      : {len(reddit_df):,}")
print(f"Null text rows  : {reddit_df['cleaned_text'].isna().sum():,}")
print(f"Avg text length : {reddit_df['cleaned_text'].dropna().str.len().mean():.0f} chars")

# --- Drop null text rows ---
reddit_df = reddit_df.dropna(subset=["cleaned_text"]).reset_index(drop=True)
print(f"\n✅ After dropping nulls: {len(reddit_df):,} rows remain")

# --- Quick sample ---
print("\n--- Sample post (anxiety) ---")
sample = reddit_df[reddit_df["source"] == "anxiety"]["cleaned_text"].iloc[0]
print(sample[:300], "...")

📊 REDDIT DATASETS
source
depression      384
mentalhealth    362
anxiety         298
therapy         170
loneliness      139
Name: count, dtype: int64

Total rows      : 1,353
Null text rows  : 0
Avg text length : 925 chars

✅ After dropping nulls: 1,353 rows remain

--- Sample post (anxiety) ---
Recruiting Moderators! Hey friends, We are looking to grow the team again here on our lovely subreddit. If you are interested, please fill out the form on our application page for r/Anxiety. If you have any questions, feel free to drop them on this post or send us a modmail. Thanks! ...


In [18]:
# ============================================================
# CELL 5 — Clean & Filter Noise
# ============================================================

# --- Noise patterns to drop (mod posts, announcements, etc.) ---
noise_keywords = [
    "recruiting moderators", "mod announcement", "subreddit rules",
    "updated rules", "bot advertisement", "automoderator",
    "weekly thread", "daily thread", "megathread"
]

def is_noise(text):
    text_lower = text.lower()
    return any(kw in text_lower for kw in noise_keywords)

# --- Apply filter ---
before = len(reddit_df)
reddit_df = reddit_df[~reddit_df["cleaned_text"].apply(is_noise)].reset_index(drop=True)
after = len(reddit_df)

print(f"  Removed {before - after} noise rows")
print(f" Clean Reddit rows: {after:,}")

# --- Also filter very short posts (less than 50 chars — likely useless) ---
reddit_df = reddit_df[reddit_df["cleaned_text"].str.len() >= 50].reset_index(drop=True)
print(f" After length filter: {len(reddit_df):,} rows remain")

# --- Combine Reddit + Counseling into master df ---
master_df = pd.concat([reddit_df, counseling_df], ignore_index=True)
master_df = master_df[["id", "source", "cleaned_text"]].copy()

print(f"\nMaster dataframe: {len(master_df):,} rows")
print(f"   Sources: {master_df['source'].value_counts().to_dict()}")

  Removed 0 noise rows
 Clean Reddit rows: 1,307
 After length filter: 1,307 rows remain

Master dataframe: 2,137 rows
   Sources: {'counseling': 830, 'depression': 364, 'mentalhealth': 359, 'anxiety': 290, 'therapy': 166, 'loneliness': 128}


In [19]:
# ============================================================
# CELL 6 — Pain Point Extraction
# ============================================================

# --- Pain point keyword map ---
pain_point_map = {
    "anxiety & panic"       : ["anxiety", "panic attack", "anxious", "overthinking", "worry", "nervous", "dread", "fear"],
    "depression"            : ["depression", "depressed", "hopeless", "empty", "numb", "worthless", "no point", "can't get up"],
    "loneliness"            : ["lonely", "loneliness", "isolated", "no friends", "no one understands", "alone", "disconnected"],
    "trauma & abuse"        : ["trauma", "abuse", "abused", "ptsd", "assault", "childhood", "neglect", "toxic"],
    "relationship issues"   : ["relationship", "boyfriend", "girlfriend", "partner", "marriage", "divorce", "breakup", "toxic relationship"],
    "self worth"            : ["worthless", "hate myself", "self esteem", "confidence", "i'm a failure", "not good enough", "ashamed"],
    "suicidal thoughts"     : ["suicidal", "suicide", "end my life", "don't want to live", "kill myself", "self harm", "cutting"],
    "sleep & exhaustion"    : ["can't sleep", "insomnia", "exhausted", "tired all the time", "no energy", "fatigue"],
    "work & burnout"        : ["burnout", "work stress", "job", "career", "fired", "overwhelmed at work", "toxic workplace"],
    "family issues"         : ["family", "parents", "mother", "father", "siblings", "toxic family", "home environment"],
}

def extract_pain_points(text):
    text_lower = text.lower()
    matched = []
    for pain_point, keywords in pain_point_map.items():
        if any(kw in text_lower for kw in keywords):
            matched.append(pain_point)
    return matched if matched else ["unclassified"]

# --- Apply ---
master_df["pain_points"] = master_df["cleaned_text"].apply(extract_pain_points)
master_df["pain_point_count"] = master_df["pain_points"].apply(len)

# --- Stats ---
print("📊 Pain point distribution:")
from collections import Counter
all_pain_points = [pp for sublist in master_df["pain_points"] for pp in sublist]
pain_counts = Counter(all_pain_points)
for pain, count in pain_counts.most_common():
    pct = count / len(master_df) * 100
    print(f"   {pain:30s} → {count:>4} rows ({pct:.1f}%)")

print(f"\n✅ Rows with 1+ pain point : {(master_df['pain_point_count'] > 0).sum():,}")
print(f"   Avg pain points per row : {master_df['pain_point_count'].mean():.2f}")
print(f"   Unclassified            : {(master_df['pain_points'].apply(lambda x: x == ['unclassified'])).sum():,}")

📊 Pain point distribution:
   unclassified                   →  648 rows (30.3%)
   anxiety & panic                →  519 rows (24.3%)
   relationship issues            →  482 rows (22.6%)
   family issues                  →  437 rows (20.4%)
   depression                     →  411 rows (19.2%)
   loneliness                     →  291 rows (13.6%)
   trauma & abuse                 →  197 rows (9.2%)
   work & burnout                 →  185 rows (8.7%)
   suicidal thoughts              →  131 rows (6.1%)
   self worth                     →  106 rows (5.0%)
   sleep & exhaustion             →   78 rows (3.6%)

✅ Rows with 1+ pain point : 2,137
   Avg pain points per row : 1.63
   Unclassified            : 648


In [ ]:
# ============================================================
# CELL 6b — Expand Keyword Map (reduce unclassified)
# ============================================================

# --- Sample unclassified posts to understand what we're missing ---
unclassified_mask = master_df["pain_points"].apply(lambda x: x == ["unclassified"])
unclassified_samples = master_df[unclassified_mask]["cleaned_text"].sample(10, random_state=42)

print("🔍 Sample unclassified posts (first 200 chars each):\n")
for i, text in enumerate(unclassified_samples, 1):
    print(f"[{i}] {text[:200]}\n")

🔍 Sample unclassified posts (first 200 chars each):

[1] How do you know you have the right therapist for you? How would I know how to "train" my therapist to be able to give me what I need from treatment?

[2] What to do now that i don't feel like i'll never be happy again? I just feel like there is anything other than a miracle that could make me happy. Im on clonazepam 0,25 daily and thats what is keeping

[3] I read that you should ignore them and they have to come to a conclusion that they were wrong on their own terms. Is that correct?

[4] For the past year, I have been feeling pressured to do well in school, and it put a ton of stress on me. I have been bullied for five years, and for some reason, it is now sinking in, and I can't stop

[5] What the heck My account got warned so no more updates i'ma just eat my pills in peace

[6] I am conflicted with this guy I work with. He is a coworker and close friend whom I've had a crush on since meeting him four years ago. We made out o

In [21]:
# ============================================================
# CELL 6c — Expanded Keyword Map + Re-run Extraction
# ============================================================

pain_point_map = {
    "anxiety & panic"       : [
        "anxiety", "panic attack", "anxious", "overthinking", "worry",
        "nervous", "dread", "fear", "stressed", "stress", "overwhelmed",
        "can't breathe", "heart racing", "restless", "on edge"
    ],
    "depression"            : [
        "depression", "depressed", "hopeless", "empty", "numb", "worthless",
        "no point", "can't get up", "don't feel anything", "feel nothing",
        "never be happy", "never happy", "what's the point", "no future",
        "drowning", "struggling to stay afloat", "just let go"
    ],
    "loneliness"            : [
        "lonely", "loneliness", "isolated", "no friends", "no one understands",
        "alone", "disconnected", "fading connections", "feel like a stranger",
        "no one to talk to", "people don't care", "invisible", "left out"
    ],
    "trauma & abuse"        : [
        "trauma", "abuse", "abused", "ptsd", "assault", "childhood",
        "neglect", "toxic", "bullied", "bullying", "harassed", "violated",
        "hurt me", "they hurt"
    ],
    "relationship issues"   : [
        "relationship", "boyfriend", "girlfriend", "partner", "marriage",
        "divorce", "breakup", "toxic relationship", "crush", "unrequited",
        "he doesn't", "she doesn't", "we broke", "coworker", "flirting",
        "made out", "rejected", "friendzone"
    ],
    "self worth"            : [
        "worthless", "hate myself", "self esteem", "confidence", "i'm a failure",
        "not good enough", "ashamed", "ugly", "stupid", "i'm the problem",
        "no one likes me", "i'm too much", "i'm not enough"
    ],
    "suicidal thoughts"     : [
        "suicidal", "suicide", "end my life", "don't want to live",
        "kill myself", "self harm", "cutting", "eat my pills", "take my pills",
        "let the water take me", "stop struggling", "let go of everything",
        "wanting to rest", "wanting to stop", "ending it", "not be here",
        "disappear forever", "give up on life"
    ],
    "sleep & exhaustion"    : [
        "can't sleep", "insomnia", "exhausted", "tired all the time",
        "no energy", "fatigue", "barely sleep", "sleep all day",
        "always tired", "drained"
    ],
    "work & burnout"        : [
        "burnout", "work stress", "job", "career", "fired", "overwhelmed at work",
        "toxic workplace", "boss", "pressure at work", "overworked"
    ],
    "academic stress"       : [
        "school", "university", "college", "exam", "grades", "study",
        "academic pressure", "failing class", "professor", "homework",
        "pressure to do well", "student"
    ],
    "family issues"         : [
        "family", "parents", "mother", "father", "siblings", "toxic family",
        "home environment", "mom", "dad", "abusive parent", "controlling parent"
    ],
    "therapy seeking"       : [
        "therapist", "therapy", "counselor", "psychiatrist", "psychologist",
        "find a therapist", "right therapist", "mental health professional",
        "looking for help", "seeking help", "need someone to talk to"
    ],
}

# --- Re-run extraction ---
master_df["pain_points"] = master_df["cleaned_text"].apply(extract_pain_points)
master_df["pain_point_count"] = master_df["pain_points"].apply(len)

# --- Stats ---
print("📊 Updated pain point distribution:")
all_pain_points = [pp for sublist in master_df["pain_points"] for pp in sublist]
pain_counts = Counter(all_pain_points)
for pain, count in pain_counts.most_common():
    pct = count / len(master_df) * 100
    print(f"   {pain:30s} → {count:>4} rows ({pct:.1f}%)")

unclassified_count = (master_df["pain_points"].apply(lambda x: x == ["unclassified"])).sum()
print(f"\n✅ Unclassified now: {unclassified_count} ({unclassified_count/len(master_df)*100:.1f}%)")

📊 Updated pain point distribution:
   anxiety & panic                →  602 rows (28.2%)
   family issues                  →  593 rows (27.7%)
   relationship issues            →  544 rows (25.5%)
   unclassified                   →  474 rows (22.2%)
   depression                     →  430 rows (20.1%)
   therapy seeking                →  327 rows (15.3%)
   loneliness                     →  307 rows (14.4%)
   academic stress                →  300 rows (14.0%)
   trauma & abuse                 →  230 rows (10.8%)
   work & burnout                 →  192 rows (9.0%)
   self worth                     →  169 rows (7.9%)
   suicidal thoughts              →  141 rows (6.6%)
   sleep & exhaustion             →   97 rows (4.5%)

✅ Unclassified now: 474 (22.2%)


In [22]:
# ============================================================
# CELL 6d — Sample remaining unclassified
# ============================================================

unclassified_mask = master_df["pain_points"].apply(lambda x: x == ["unclassified"])
unclassified_samples = master_df[unclassified_mask]["cleaned_text"].sample(15, random_state=99)

print(f"🔍 Remaining unclassified: {unclassified_mask.sum()} posts\n")
for i, text in enumerate(unclassified_samples, 1):
    print(f"[{i}] {text[:200]}\n")

🔍 Remaining unclassified: 474 posts

[1] [ Removed by Reddit ] Removed by Reddit on account of violating the [content policy. ]

[2] She cried because she lost all trust in me. Now I feel disgusting and like my heart just got stabbed, what do I do?

[3] Yes, this is a very sensible sticky in my opinion. I have been on both sides of this coin. As a "helper", I find that I didn't really help the person via a more live chatting or back and forth messagi

[4] Fuck everything man It gets better for one fucking hour and then I feel like someone sucked the life out of me im so fucking done

[5] It has been going on more often lately, but not all the time. It has been starting to affect my driving, among other things.

[6] I need answers to my anger, possessiveness, and urges. I am angry all the time. I push people away so much that I actually blow up on people now because they ask me how my day is or was. I have a poss

[7] I'm just an extremely paranoid person. I constantly feel as though I 

In [23]:
# ============================================================
# CELL 6e — Final Keyword Expansion + Drop Remaining Noise
# ============================================================

# --- Drop true noise (reddit meta posts) ---
noise_patterns = [
    "removed by reddit",
    "violating the content policy",
    "i am a bot",
    "this is a sticky",
    "wanted to apologize to the mods",
    "if i see someone struggling i'll attempt a dm",
]

def is_noise(text):
    text_lower = text.lower()
    return any(p in text_lower for p in noise_patterns)

before = len(master_df)
master_df = master_df[~master_df["cleaned_text"].apply(is_noise)].reset_index(drop=True)
print(f"🗑️  Dropped {before - len(master_df)} noise rows → {len(master_df):,} remain")

# --- Add new pain point categories ---
pain_point_map.update({
    "anger & emotional dysregulation" : [
        "angry all the time", "anger", "rage", "furious", "blow up",
        "push people away", "mood swings", "explosive", "i snapped",
        "i lashed out", "i yelled", "can't control my emotions",
        "feel disgusting", "heart got stabbed"
    ],
    "intrusive thoughts & paranoia"   : [
        "intrusive thoughts", "paranoid", "paranoia", "violent thoughts",
        "nightmare", "nightmares", "horrifying thoughts", "unwanted thoughts",
        "obsessive thoughts", "can't stop thinking", "horror", "in danger",
        "someone is watching", "worsens at night"
    ],
    "mood disorders"                  : [
        "bipolar", "mood disorder", "mania", "manic episode", "mood swings",
        "undiagnosed", "borderline", "bpd", "dissociation", "episodes"
    ],
    "physical symptoms"               : [
        "headache", "chest pain", "can't breathe", "heart racing",
        "clenching", "tension", "body aches", "nausea", "stomach",
        "physical symptoms", "pain in my body", "shaking"
    ],
    "emotional crisis"                : [
        "so fucking done", "i give up", "i can't do this anymore",
        "breaking point", "i'm done", "sucked the life out of me",
        "i can't take it", "i'm falling apart", "i'm losing it",
        "everything is falling apart", "i don't know what to do anymore"
    ],
})

# --- Re-run extraction ---
master_df["pain_points"] = master_df["cleaned_text"].apply(extract_pain_points)
master_df["pain_point_count"] = master_df["pain_points"].apply(len)

# --- Stats ---
print("\n📊 Final pain point distribution:")
all_pain_points = [pp for sublist in master_df["pain_points"] for pp in sublist]
pain_counts = Counter(all_pain_points)
for pain, count in pain_counts.most_common():
    pct = count / len(master_df) * 100
    print(f"   {pain:35s} → {count:>4} rows ({pct:.1f}%)")

unclassified_count = (master_df["pain_points"].apply(lambda x: x == ["unclassified"])).sum()
print(f"\n✅ Unclassified now: {unclassified_count} ({unclassified_count/len(master_df)*100:.1f}%)")

🗑️  Dropped 6 noise rows → 2,131 remain

📊 Final pain point distribution:
   anxiety & panic                     →  602 rows (28.2%)
   family issues                       →  593 rows (27.8%)
   relationship issues                 →  544 rows (25.5%)
   depression                          →  430 rows (20.2%)
   unclassified                        →  422 rows (19.8%)
   therapy seeking                     →  327 rows (15.3%)
   loneliness                          →  306 rows (14.4%)
   academic stress                     →  300 rows (14.1%)
   trauma & abuse                      →  229 rows (10.7%)
   work & burnout                      →  192 rows (9.0%)
   self worth                          →  169 rows (7.9%)
   anger & emotional dysregulation     →  157 rows (7.4%)
   suicidal thoughts                   →  140 rows (6.6%)
   sleep & exhaustion                  →   97 rows (4.6%)
   physical symptoms                   →   81 rows (3.8%)
   intrusive thoughts & paranoia       →   79 r

In [24]:
# ============================================================
# CELL 6f — Accept & Relabel Unclassified
# ============================================================

def relabel_unclassified(pain_points):
    if pain_points == ["unclassified"]:
        return ["general mental health"]
    return pain_points

master_df["pain_points"] = master_df["pain_points"].apply(relabel_unclassified)

# --- Final check ---
print("✅ Pain point extraction complete")
print(f"   Total rows          : {len(master_df):,}")
print(f"   Unique categories   : {len(set(pp for sublist in master_df['pain_points'] for pp in sublist))}")
print(f"   Avg pain points/row : {master_df['pain_point_count'].mean():.2f}")
print(f"   Multi-label rows    : {(master_df['pain_point_count'] > 1).sum():,} ({(master_df['pain_point_count'] > 1).mean()*100:.1f}%)")

✅ Pain point extraction complete
   Total rows          : 2,131
   Unique categories   : 18
   Avg pain points/row : 2.24
   Multi-label rows    : 1,097 (51.5%)


In [25]:
# ============================================================
# CELL 6f — Accept & Relabel Unclassified
# ============================================================

def relabel_unclassified(pain_points):
    if pain_points == ["unclassified"]:
        return ["general mental health"]
    return pain_points

master_df["pain_points"] = master_df["pain_points"].apply(relabel_unclassified)

# --- Final check ---
print("✅ Pain point extraction complete")
print(f"   Total rows          : {len(master_df):,}")
print(f"   Unique categories   : {len(set(pp for sublist in master_df['pain_points'] for pp in sublist))}")
print(f"   Avg pain points/row : {master_df['pain_point_count'].mean():.2f}")
print(f"   Multi-label rows    : {(master_df['pain_point_count'] > 1).sum():,} ({(master_df['pain_point_count'] > 1).mean()*100:.1f}%)")

✅ Pain point extraction complete
   Total rows          : 2,131
   Unique categories   : 18
   Avg pain points/row : 2.24
   Multi-label rows    : 1,097 (51.5%)


In [26]:
# ============================================================
# CELL 6g — Group Pain Points into Macro Categories
# ============================================================

macro_map = {
    "emotional distress"      : [
        "depression", "emotional crisis", "mood disorders", "self worth"
    ],
    "anxiety & fear"          : [
        "anxiety & panic", "intrusive thoughts & paranoia", "physical symptoms"
    ],
    "social & relationships"  : [
        "loneliness", "relationship issues", "family issues",
        "anger & emotional dysregulation"
    ],
    "trauma & crisis"         : [
        "trauma & abuse", "suicidal thoughts"
    ],
    "life stressors"          : [
        "work & burnout", "academic stress", "sleep & exhaustion"
    ],
    "help seeking"            : [
        "therapy seeking", "general mental health"
    ],
}

# --- Reverse map: sub-category → macro ---
sub_to_macro = {}
for macro, subs in macro_map.items():
    for sub in subs:
        sub_to_macro[sub] = macro

def assign_macro(pain_points):
    macros = list(set(sub_to_macro.get(pp, "help seeking") for pp in pain_points))
    return macros

master_df["macro_groups"] = master_df["pain_points"].apply(assign_macro)
master_df["primary_macro"] = master_df["macro_groups"].apply(lambda x: x[0] if len(x) == 1 else
    # if multiple, pick by priority order
    next((m for m in ["trauma & crisis", "emotional distress", "anxiety & fear",
                       "social & relationships", "life stressors", "help seeking"]
          if m in x), x[0]))

# --- Stats ---
print("📊 Macro group distribution:")
macro_counts = master_df["primary_macro"].value_counts()
for macro, count in macro_counts.items():
    pct = count / len(master_df) * 100
    bar = "█" * int(pct / 2)
    print(f"   {macro:25s} → {count:>4} ({pct:.1f}%) {bar}")

📊 Macro group distribution:
   social & relationships    →  489 (22.9%) ███████████
   help seeking              →  472 (22.1%) ███████████
   emotional distress        →  407 (19.1%) █████████
   anxiety & fear            →  379 (17.8%) ████████
   trauma & crisis           →  340 (16.0%) ███████
   life stressors            →   44 (2.1%) █


In [27]:
# ============================================================
# CELL 7 — Age Extraction with spaCy (NLP)
# ============================================================

# --- Life stage keyword → age range (for when no number is found) ---
life_stage_map = [
    (["middle school", "middle schooler"],                              (11, 14)),
    (["high school", "high schooler", "high school student"],           (15, 18)),
    (["college", "university", "freshman", "sophomore",
      "junior year", "senior year"],                                    (18, 23)),
    (["just graduated", "recent graduate", "entry level", "first job"], (22, 27)),
    (["my kids", "my children", "my son", "my daughter",
      "as a parent", "i'm a parent"],                                   (28, 45)),
    (["my teenage", "my teen"],                                         (30, 48)),
    (["just retired", "after retirement", "i retired"],                 (60, 70)),
    (["my grandchildren", "my grandkids"],                              (55, 75)),
]

def extract_age_nlp(text):
    doc = nlp(text[:1000])  # limit to first 1000 chars for speed

    # --- Step 1: use spaCy to find cardinal numbers near age context words ---
    age_context_words = {"years", "year", "old", "age", "aged", "turning", "turned"}

    for token in doc:
        if token.like_num:
            try:
                age = int(token.text)
            except ValueError:
                continue
            if not (13 <= age <= 80):
                continue

            # check surrounding tokens for age context
            window = [t.text.lower() for t in doc[max(0, token.i-3):token.i+4]]
            if any(w in age_context_words for w in window):
                return age, "nlp_cardinal"

    # --- Step 2: spaCy entities — look for CARDINAL near "years old" ---
    for ent in doc.ents:
        if ent.label_ == "CARDINAL":
            try:
                age = int(ent.text)
            except ValueError:
                continue
            if not (13 <= age <= 80):
                continue
            context = text[max(0, ent.start_char-20):ent.end_char+20].lower()
            if any(w in context for w in ["year", "old", "age", "yo"]):
                return age, "nlp_entity"

    # --- Step 3: gender+age pattern e.g. "22F" "19M" using spaCy tokens ---
    for token in doc:
        if token.text[:-1].isdigit() and token.text[-1].lower() in ("f", "m"):
            try:
                age = int(token.text[:-1])
                if 13 <= age <= 80:
                    return age, "nlp_gender_tag"
            except ValueError:
                continue

    # --- Step 4: life stage inference ---
    text_lower = text.lower()
    for keywords, (age_min, age_max) in life_stage_map:
        if any(kw in text_lower for kw in keywords):
            return int((age_min + age_max) / 2), "life_stage"

    return None, "unknown"

# --- Apply (with progress indicator) ---
print("⏳ Running spaCy NLP on 2,131 posts...")

results = master_df["cleaned_text"].apply(lambda x: pd.Series(extract_age_nlp(str(x))))
master_df[["age_est", "age_method"]] = results

print("✅ Done")

# --- Age bins ---
def age_to_group(age):
    if age is None:
        return "unknown"
    if age < 18:
        return "teen (< 18)"
    elif age <= 25:
        return "young adult (18–25)"
    elif age <= 35:
        return "adult (26–35)"
    elif age <= 50:
        return "mid adult (36–50)"
    else:
        return "older adult (50+)"

master_df["age_group"] = master_df["age_est"].apply(age_to_group)

# --- Stats ---
print("\n📊 Age extraction method breakdown:")
print(master_df["age_method"].value_counts().to_string())

print("\n📊 Age group distribution:")
age_counts = master_df["age_group"].value_counts()
for group, count in age_counts.items():
    pct = count / len(master_df) * 100
    bar = "█" * int(pct / 2)
    print(f"   {group:25s} → {count:>4} ({pct:.1f}%) {bar}")

# --- Sanity check ---
print("\n🔍 Sample extractions:")
sample = master_df[master_df["age_method"] != "unknown"][["cleaned_text", "age_est", "age_method"]].head(5)
for _, row in sample.iterrows():
    print(f"   [{row['age_method']:15s}] Age {row['age_est']:>2} | {row['cleaned_text'][:100]}")

⏳ Running spaCy NLP on 2,131 posts...
✅ Done

📊 Age extraction method breakdown:
age_method
unknown           1837
life_stage         150
nlp_cardinal       106
nlp_gender_tag      28
nlp_entity          10

📊 Age group distribution:
   older adult (50+)         → 1839 (86.3%) ███████████████████████████████████████████
   young adult (18–25)       →  126 (5.9%) ██
   teen (< 18)               →   81 (3.8%) █
   mid adult (36–50)         →   52 (2.4%) █
   adult (26–35)             →   33 (1.5%) 

🔍 Sample extractions:
   [nlp_cardinal   ] Age 25.0 | to any anxious moms out there hi moms! ive been thinking about the future as a 25 year old wanting k
   [nlp_cardinal   ] Age 24.0 | Struggling to breathe for a month Good day everyone, I am a 24 year old male. I write this post out 
   [nlp_cardinal   ] Age 34.0 | It's my birthday and I feel awful I'm turning 34 today, and I'm feeling a lot of things, but mostly 
   [nlp_gender_tag ] Age 25.0 | Shrooms After Effects Has anyone had any per